# 1. Random SO

In [13]:
import pandas as pd
import random
import numpy as np
from datetime import datetime, timedelta

# Set seed for reproducibility
np.random.seed(42)
random.seed(42)

# Item codes list
item_codes = [
    'HBG-420', 'HBG-440', 'HBG-460', 'HBM-740', 'ACS-165', 'AST-055', 'AST-070', 
    'ABB-068', 'ABB-073', 'CCS-170', 'GRB-003', 'GRB-005', 'GRB-007', 'GRF-003', 
    'GRF-005', 'GRF-007', 'GRK-000', 'RIM-127', 'RIM-427', 'RIM-429', 'RIM-700', 
    'TRB-403', 'TRB-405', 'TRB-407', 'TRF-403', 'TRF-405', 'TRF-407', 'WLS-127', 
    'WLS-427', 'WLS-429', 'WLS-723', 'XCB-103', 'XCB-105', 'XCB-107', 'XCF-103', 
    'XCF-105', 'XCF-107', 'ASL-110', 'ASL-140', 'ASL-170', 'ENB-703', 'ENB-705', 
    'ENB-707', 'ENF-703', 'ENF-705', 'ENF-707', 'HBM-750', 'HBM-800', 'ACR-032', 
    'ACR-034', 'ACR-036', 'ACR-038', 'ALB-754', 'ALB-756', 'ALB-758', 'AXL-150', 
    'BBB-030', 'BBS-068', 'BBS-073', 'BRK-134', 'BRK-135', 'BRK-202'
]

# Generate item selection probabilities based on normal distribution
# Mean = 50 (middle of 63 items), Std = 10
num_items = len(item_codes)
item_indices = np.arange(num_items)

# Generate probabilities from normal distribution centered at middle
mean_idx = 100 * (num_items / 100)  # Scale mean to actual item count #50
std_idx = 15 * (num_items / 100)   # Scale std to actual item count #10

# Create normal distribution weights
weights = np.exp(-0.5 * ((item_indices - mean_idx) / std_idx) ** 2)
probabilities = weights / weights.sum()

# Generate orders
orders = []
order_counter = 1

# Due date range
start_due_date = datetime(2023, 2, 1)
end_due_date = datetime(2023, 3, 1)

# Generate approximately 100 orders
num_orders = 50

for i in range(num_orders):
    # Random due date within range
    days_offset = random.randint(0, (end_due_date - start_due_date).days)
    due_date = start_due_date + timedelta(days=days_offset)
    
    # Order date is typically 10-20 days before due date (lead time simulation)
    lead_time = random.randint(10, 20)
    order_date = due_date - timedelta(days=lead_time)
    
    # Generate OrderID based on order_date
    order_id = f"SO-{order_date.strftime('%y%m%d')}-{random.randint(10,19):02d}-{order_counter:02d}"
    
    # Select item using normal distribution probabilities
    item_code = np.random.choice(item_codes, p=probabilities)
    
    # Quantity: normal distribution with mean=50, std=10
    qty_raw = np.random.normal(loc=50, scale=10)
    # Round to integer and ensure minimum quantity of 1
    qty = max(1, int(round(qty_raw)))
    
    orders.append({
        'OrderDate': order_date.strftime('%Y-%m-%d'),
        'OrderID': order_id,
        'ItemCode': item_code,
        'Qty': qty,
        'DueDate': due_date.strftime('%Y-%m-%d')
    })
    
    order_counter += 1
    if order_counter > 99:
        order_counter = 1

# Create DataFrame and sort by OrderDate, then OrderID
df = pd.DataFrame(orders)
df = df.sort_values(['OrderDate', 'OrderID']).reset_index(drop=True)

# Analytics summary
print("="*70)
print("SALES ORDER DATA - GENERATION SUMMARY")
print("="*70)
print(f"Total orders generated: {len(df)}")
print(f"Unique items ordered: {df['ItemCode'].nunique()} out of {len(item_codes)}")
print(f"Order date range: {df['OrderDate'].min()} to {df['OrderDate'].max()}")
print(f"Due date range: {df['DueDate'].min()} to {df['DueDate'].max()}")
print(f"\nQuantity distribution:\n{df['Qty'].value_counts().sort_index()}")

# Item frequency analysis
print(f"\nTop 10 most ordered items:")
item_freq = df['ItemCode'].value_counts().head(10)
for item, count in item_freq.items():
    print(f"  {item}: {count} orders")

print("\n" + "="*70)
print("FULL CSV OUTPUT")
print("="*70)
csv_output = df.to_csv("random_SO.csv",index=False)
print(csv_output)

SALES ORDER DATA - GENERATION SUMMARY
Total orders generated: 50
Unique items ordered: 18 out of 62
Order date range: 2023-01-14 to 2023-02-15
Due date range: 2023-02-01 to 2023-03-01

Quantity distribution:
Qty
24    1
34    1
35    2
36    1
37    1
39    1
40    3
41    2
43    1
44    4
45    3
46    3
48    4
49    2
51    1
52    1
53    3
54    1
55    1
56    2
59    3
60    3
61    1
62    3
65    1
78    1
Name: count, dtype: int64

Top 10 most ordered items:
  ALB-754: 5 orders
  ACR-034: 5 orders
  BBS-073: 5 orders
  BRK-135: 5 orders
  BBB-030: 4 orders
  BRK-134: 3 orders
  AXL-150: 3 orders
  BRK-202: 3 orders
  BBS-068: 3 orders
  ENF-707: 2 orders

FULL CSV OUTPUT
None


# 2. Separate input file

In [1]:
import pandas as pd
import os

FILE = r"..\test_excel\data_generation(ALL_input_here_first).xlsx"

EXPORT_FOLDER = r"..\data\from_PS"

# --- Ensure folder exists ---
os.makedirs(EXPORT_FOLDER, exist_ok=True)

# LOAD WORKBOOK - create an excel object (before reading all sheet names)
xls = pd.ExcelFile(FILE)
print(xls.sheet_names)

# --- Loop through each sheet ---
for sh_name in xls.sheet_names:
    # read sheet into dataframe
    df = pd.read_excel(xls, sheet_name= sh_name)
    
    # Build clean CSV filename
    txt_filename = f"{sh_name.strip().lower().replace(' ', '_')}.txt"
    txt_path = os.path.join(EXPORT_FOLDER, txt_filename)
    
    # Write DataFrame to txt
    
    df.to_csv(txt_path, sep='\t', index=False, encoding='utf-8-sig')        
    #CSV OPTION df.to_csv(csv_path, index=False, encoding='utf-8-sig')
    
    print(f"✅ Exported: {txt_path}")
    
# --- CLOSE workbook to release file lock ---
xls.close()

['item_master', 'policy_master', 'bom_master', 'onhand', 'demand_orders', 'supply_orders']
✅ Exported: ..\data\from_PS\item_master.txt
✅ Exported: ..\data\from_PS\policy_master.txt
✅ Exported: ..\data\from_PS\bom_master.txt
✅ Exported: ..\data\from_PS\onhand.txt
✅ Exported: ..\data\from_PS\demand_orders.txt
✅ Exported: ..\data\from_PS\supply_orders.txt
